# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzaibazhar895-bit/FlyRank_ai_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


Unit of Analysis: One row represents one content item (page) belonging to one client. For this assignment I am analyzing content performance and refresh opportunities at the page level.

Time Window: I am using the March 2026 slice of the warehouse dataset (month = 2026-03) for exploration and feature development. This avoids using the latest month and reduces the risk of future information leakage.

Goal: Rank content pages that may deserve review, refresh, or monitoring based on observed search and engagement signals.

Tables Used: I am using the fact_content_daily_performance table because it contains daily search and engagement metrics needed for content performance analysis.

In [3]:
!pip install -q duckdb duckdb-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 670.9 kB/s eta 0:00:00


In [7]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("Flyrank_ai_Intern")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

Connected successfully!


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────────┬────────────┬────────────┐
│  rows   │ unique_content │ start_date │  end_date  │
│  int64  │     int64      │    date    │    date    │
├─────────┼────────────────┼────────────┼────────────┤
│ 9841378 │         331437 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions gsc_clicks gsc_avg_position ga4_sessions ga4_engaged_sessions

Label / Proxy: Content review opportunity proxy.

For this exercise, pages with fewer than 5 GA4 sessions are treated as higher review-priority pages. This is a defined rule for exploration and not a proven future outcome.

Context Fields: client_hash_id report_date

Excluded Fields: content_hash_id because it is only an identifier and does not help prediction.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
gsc_impressions,
gsc_clicks,
gsc_avg_position,
ga4_sessions,
ga4_engaged_sessions
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────┬──────────────────┬──────────────┬──────────────────────┐
│ gsc_impressions │ gsc_clicks │ gsc_avg_position │ ga4_sessions │ ga4_engaged_sessions │
│      int64      │   int64    │      double      │    int64     │        int64         │
├─────────────────┼────────────┼──────────────────┼──────────────┼──────────────────────┤
│               0 │          0 │             NULL │            1 │                    0 │
│               0 │          0 │             NULL │            1 │                    0 │
│               0 │          0 │             NULL │            1 │                    0 │
│               0 │          0 │             NULL │            1 │                    0 │
│               0 │          0 │             NULL │            1 │                    0 │
└─────────────────┴────────────┴──────────────────┴──────────────┴──────────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Feature 1: gsc_impressions Knowable at the decision moment because impressions are historical search performance measurements.

Feature 2: gsc_clicks Knowable at the decision moment because clicks are recorded before any content review decision.

Feature 3: gsc_avg_position Knowable at the decision moment because it reflects past search ranking performance.

Feature 4: ga4_sessions Knowable at the decision moment because session data already exists before the review process.

Feature 5: ga4_engaged_sessions Knowable at the decision moment because engagement is measured from previous user interactions.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

sample = con.sql(f"""
SELECT
gsc_impressions,
gsc_clicks,
gsc_avg_position,
ga4_sessions
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
LIMIT 1000
""").df()

# Simple proxy label
sample["label"] = (sample["ga4_sessions"] < 5).astype(int)

# Deliberate leakage
sample["leak_feature"] = sample["label"]

sample.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,label,leak_feature
0,0,0,NaN,1,1,1
1,0,0,NaN,1,1,1
2,0,0,NaN,1,1,1
3,0,0,NaN,1,1,1
4,0,0,NaN,1,1,1


Leakage Experiment

I intentionally created a feature called leak_feature by copying the label.

This feature contains the answer the model is trying to predict, so it would create unrealistically high performance.

The feature would be removed before real modeling.

In [12]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [13]:
con.sql(f"""
SELECT
COUNT(*) AS row_count,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

┌───────────┬────────────┬────────────┐
│ row_count │ start_date │  end_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [14]:
con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Client history is unbalanced because different clients have different amounts of historical data.

Some records contain Search Console data but do not contain GA4 data.

This dataset can support observed and directional analysis, but it cannot prove that a content refresh caused performance changes.

The dataset is pseudonymized, so real URLs, keywords, and client identities are unavailable.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.